# Module 1 hook demo

Instructor only. Not linked from the student-facing site.

Used live during **`lectures/dsca-module-01.html`**, slides 2 ("Before we watch this") and 3 ("The five stages, already in front of you"). The lecture's own speaker notes for slide 2 say to switch to this notebook; this cell is the notebook side of that link.

Run this once before Module 1, with a real connection (a phone hotspot is fine), and save the notebook without clearing output. If the room's connection has a bad moment live, this morning's saved output is already on screen to narrate. If the connection holds, re-run the reference agent cell live for the real "watch it happen now" moment.

The naive bot needs no connection at all, it is a plain function. Only the reference agent cell calls a live API.

**First time running this notebook?** Do the one-time setup below before anything else.

## One-time setup

Do this once per machine, before running any cell below.

1. From the repository root, run `cd demos`, create the shared environment with `python3 -m venv .venv`, then activate it with `source .venv/bin/activate`. Select `demos/.venv` as this notebook's kernel (in VS Code: the kernel picker top right; in Jupyter: `New > Python 3` or select an existing kernel).
2. Run the cell immediately below once, in that environment.
3. Copy `demos/.env.example` to `demos/.env` and fill in `GOOGLE_API_KEY`. See `demos/README.md` for exactly how to get one, it is free and needs no card.

See `demos/README.md` for the full setup story and the one-folder-per-module convention this notebook lives under.

In [ ]:
%pip install -r ../requirements.txt


## Part 1: the naive bot

No API, no internet. Deterministic every time, safe to run live regardless of connectivity.

In [ ]:
def naive_bot(turns):
    """A keyword-matching bot with no memory of its own prior turns.
    Each call only sees the current line, nothing before it."""
    booking = {}
    replies = []
    for turn in turns:
        words = turn.lower().split()
        if "book" in words or "table" in words:
            # crude slot fill from the current line only
            booking = {"party_size": None, "day": None, "time": None}
            for w in words:
                if w.isdigit():
                    booking["party_size"] = w
                if w in ("monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"):
                    booking["day"] = w
                if "pm" in w or "am" in w:
                    booking["time"] = w
            replies.append(f"Booked: {booking}")
        elif "actually" in words or "instead" in words or "make that" in turn.lower():
            # no memory of the earlier booking, so this line alone means nothing to it
            replies.append("Sorry, I didn't understand that. Could you start your booking again?")
        else:
            replies.append("Sorry, I didn't understand that.")
    return replies

conversation = [
    "Book me a table for 4 on Monday at 7pm",
    "actually, make that Tuesday",
]

for turn, reply in zip(conversation, naive_bot(conversation)):
    print(f"user: {turn}")
    print(f"bot:  {reply}\n")

The second line loses the whole booking. There is no state to correct, because there was never any state to begin with, just a reaction to whatever line came in last.

## Part 2: the reference agent

Same two lines. This one holds state across turns and updates only the field that changed. Needs `GOOGLE_API_KEY` set in `.env`, and a live connection for this cell only.

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Notebook tools choose different working folders. Find the one shared
# demos/.env whether this notebook was opened from the repo, demos, or here.
env_candidates = (Path.cwd() / ".env", Path.cwd() / "demos/.env", Path.cwd().parent / ".env")
demo_env = next((path for path in env_candidates if path.is_file()), None)
if demo_env is None:
    raise FileNotFoundError("Create demos/.env from demos/.env.example before continuing.")
load_dotenv(demo_env)

# Read from demos/.env so an instructor can point at a newer or different
# model without editing this notebook if Gemini retires or replaces this
# one; "gemini-3.8-flash" is this course's own established default,
# matching every other module's demo.
GENERATION_MODEL = os.getenv("GENERATION_MODEL", "gemini-3.8-flash")

# The three fields this demo tracks across turns. Kept as one dict so the
# schema below and the starting state can't drift out of sync with each other.
FIELDS = {
    "party_size": {"type": ["integer", "null"]},
    "day": {"type": ["string", "null"]},
    "time": {"type": ["string", "null"]},
}


def reference_agent(turns):
    """Hold one running state object across turns, using Gemini's structured
    output to update only the fields the user actually mentioned each time.

    Unlike naive_bot above, the current state is part of every prompt, so a
    correction updates one field without losing the others."""
    client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
    schema = {"type": "object", "properties": FIELDS, "required": list(FIELDS)}

    state = {"party_size": None, "day": None, "time": None}
    replies = []
    for turn in turns:
        prompt = (
            f"Current booking state: {json.dumps(state)}\n"
            f"User just said: \"{turn}\"\n"
            "Return the updated booking state. Keep any field the user "
            "did not just change."
        )
        interaction = client.interactions.create(
            model=GENERATION_MODEL,
            input=prompt,
            response_format={"type": "text", "mime_type": "application/json", "schema": schema},
        )
        state = json.loads(interaction.output_text)
        replies.append(f"Got it: {state}")
    return replies


for turn, reply in zip(conversation, reference_agent(conversation)):
    print(f"user: {turn}")
    print(f"bot:  {reply}\n")


Same correction, and the day updates while the party size and time carry forward untouched. That difference, state that persists and updates instead of a bot that reacts to one line at a time, is the whole course in miniature.

Next slide: the five-stage arc (`.lu-pipeline`), naming which module builds each stage of what was just shown.